Cell 1 — imports & paths

In [9]:
import sys, os, glob, json, time
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import h5py
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# project imports (your repo)
from src.ks import run_sim
from src.io_utils import make_tag
from src.ic_utils import make_ic 

RAW_DIR = "../data/raw"
FIGS_DIR = "../figs"
RESULTS_DIR = "../results"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Dirs ready:", RAW_DIR, FIGS_DIR, RESULTS_DIR)


Dirs ready: ../data/raw ../figs ../results


Cell 2 — configuration

In [10]:
# ======= GLOBAL SIM SETTINGS =======
L, N      = 100.0, 200
steps     = 2000          # small set for quick SINDy
save_stride = 1           # IMPORTANT: save EVERY step
seed      = 1

# ======= VARIATIONS (edit to change coverage) =======
dt_list         = [0.05, 0.10]                       # temporal sampling
ic_list         = ["sin16", "gaussian", "multi_sine"]# initial conditions
noise_kind_list = ["gaussian", "uniform01", "laplace"] # noise types
noise_std_list  = [0.00, 0.02, 0.05, 0.10]           # noise levels (IC noise)

# You can trim for speed, e.g.
# noise_kind_list = ["gaussian"]
# noise_std_list  = [0.00, 0.02, 0.05]

print("Planned runs:", len(dt_list)*len(ic_list)*len(noise_kind_list)*len(noise_std_list))


Planned runs: 72


Cell 3 — generate datasets

In [11]:
import jax.numpy as jnp

def save_h5_compressed(path, u, x, t, attrs=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with h5py.File(path, "w") as f:
        f.create_dataset("u", data=np.asarray(u, dtype=np.float32),
                         compression="gzip", compression_opts=4, chunks=True)
        f.create_dataset("x", data=np.asarray(x, dtype=np.float32))
        f.create_dataset("t", data=np.asarray(t, dtype=np.float32))
        if attrs:
            for k,v in attrs.items():
                f.attrs[k] = v

count = 0
for dt in dt_list:
    for ic_kind in ic_list:
        for noise_kind in noise_kind_list:
            for noise_std in noise_std_list:
                count += 1
                x = jnp.linspace(0.0, L, N, endpoint=False)
                u0 = make_ic(x, kind=ic_kind, noise_std=noise_std,
                             noise_kind=noise_kind, seed=seed)
                trj, X, T = run_sim(L=L, N=N, dt=dt, steps=steps, u0=u0, save_stride=save_stride)

                # tag includes stride + noise kind
                tag = (make_tag(L, N, dt, ic_kind, noise_std, seed, stride=save_stride)
                       + f"_{noise_kind}")
                h5_path = f"{RAW_DIR}/{tag}.h5"
                save_h5_compressed(h5_path, trj, X, T, attrs={
                    "L":L,"N":N,"dt":dt,"steps":steps,"save_stride":save_stride,
                    "ic_kind":ic_kind,"noise_std":noise_std,"noise_kind":noise_kind,"seed":seed
                })
                print(f"[{count}] saved {tag}")
print("Done generating datasets.")


[1] saved L100_N200_dt0.050_s1_sin16_noise0.000_seed1_gaussian
[2] saved L100_N200_dt0.050_s1_sin16_noise0.020_seed1_gaussian
[3] saved L100_N200_dt0.050_s1_sin16_noise0.050_seed1_gaussian
[4] saved L100_N200_dt0.050_s1_sin16_noise0.100_seed1_gaussian
[5] saved L100_N200_dt0.050_s1_sin16_noise0.000_seed1_uniform01
[6] saved L100_N200_dt0.050_s1_sin16_noise0.020_seed1_uniform01
[7] saved L100_N200_dt0.050_s1_sin16_noise0.050_seed1_uniform01
[8] saved L100_N200_dt0.050_s1_sin16_noise0.100_seed1_uniform01
[9] saved L100_N200_dt0.050_s1_sin16_noise0.000_seed1_laplace
[10] saved L100_N200_dt0.050_s1_sin16_noise0.020_seed1_laplace
[11] saved L100_N200_dt0.050_s1_sin16_noise0.050_seed1_laplace
[12] saved L100_N200_dt0.050_s1_sin16_noise0.100_seed1_laplace
[13] saved L100_N200_dt0.050_s1_gaussian_noise0.000_seed1_gaussian
[14] saved L100_N200_dt0.050_s1_gaussian_noise0.020_seed1_gaussian
[15] saved L100_N200_dt0.050_s1_gaussian_noise0.050_seed1_gaussian
[16] saved L100_N200_dt0.050_s1_gaussian

Cell 4 — derivative builders (spectral in space, central in time)

In [12]:
def spectral_derivs(u, x):
    """
    u: (T, N) real
    returns dict of arrays (T, N): ux, uxx, uxxxx
    """
    u = np.asarray(u, dtype=float)
    T, N = u.shape
    dx = x[1]-x[0]
    freqs = np.fft.rfftfreq(N, d=dx)
    k = 2*np.pi*freqs
    ik = 1j * k
    k2 = -(k**2)        # second derivative multiplies by -(k^2)
    k4 = (k**4)         # fourth derivative multiplies by +(k^4)

    ux = np.empty_like(u)
    uxx = np.empty_like(u)
    uxxxx = np.empty_like(u)

    for i in range(T):
        uhat = np.fft.rfft(u[i])
        ux[i]    = np.fft.irfft(ik * uhat, n=N)
        uxx[i]   = np.fft.irfft(k2 * uhat, n=N)
        uxxxx[i] = np.fft.irfft(k4 * uhat, n=N)
    return {"ux": ux, "uxx": uxx, "uxxxx": uxxxx}

def time_derivative(u, dt):
    """
    central difference over time; returns ut for frames 1..T-2 (trim ends)
    u: (T,N) -> ut: (T-2, N)
    """
    u = np.asarray(u, dtype=float)
    return (u[2:] - u[:-2]) / (2*dt)

def build_features(u, x, dt):
    """
    Build regression data:
      y = ut.flatten()
      Theta columns: [u_x, u_xx, u_xxxx, u*u_x]
    Shapes -> y:(M,), Theta:(M,4), M=(T-2)*N
    """
    u = np.asarray(u, dtype=float)
    x = np.asarray(x, dtype=float)
    T, N = u.shape
    derivs = spectral_derivs(u, x)
    ut  = time_derivative(u, dt)      # (T-2, N)
    ux  = derivs["ux"][1:-1]          # align with ut
    uxx = derivs["uxx"][1:-1]
    u4  = derivs["uxxxx"][1:-1]
    uu1 = (u[1:-1] * ux)              # u * u_x

    y = ut.reshape(-1)
    Theta = np.column_stack([
        ux.reshape(-1),      # idx 0 -> u_x
        uxx.reshape(-1),     # idx 1 -> u_xx
        u4.reshape(-1),      # idx 2 -> u_xxxx
        uu1.reshape(-1),     # idx 3 -> u*u_x
    ])
    names = ["u_x", "u_xx", "u_xxxx", "u*u_x"]
    return y, Theta, names


Cell 5 — SINDy core (STLSQ), metrics, and rollout

In [13]:
def stlsq(Theta, y, thresh=1e-3, max_iter=10):
    """
    Sequential Thresholded Least Squares (basic SINDy).
    Returns xi: (n_features,)
    """
    xi, *_ = np.linalg.lstsq(Theta, y, rcond=None)
    for _ in range(max_iter):
        small = np.abs(xi) < thresh
        if small.all():
            kmax = np.argmax(np.abs(xi))
            small[kmax] = False
        active = ~small
        Xi_active, *_ = np.linalg.lstsq(Theta[:, active], y, rcond=None)
        xi = np.zeros_like(xi)
        xi[active] = Xi_active
    return xi

def support_metrics(xi, xi_true):
    eps = 1e-12
    s_hat = np.abs(xi) > 0
    s_true = np.abs(xi_true) > 0
    tp = int(np.logical_and(s_hat, s_true).sum())
    fp = int(np.logical_and(s_hat, ~s_true).sum())
    fn = int(np.logical_and(~s_hat, s_true).sum())
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1 = 2*prec*rec / (prec + rec + eps)
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

def coeff_errors(xi, xi_true):
    diff = np.abs(xi - xi_true)
    return dict(l1=float(diff.sum()), lmax=float(diff.max()))

# rollout with learned PDE (ETD1 like your solver)
class KS_LearnedStepper:
    def __init__(self, L, N, dt, coeffs):
        self.L, self.N, self.dt = L, N, dt
        self.c  = coeffs.get("u*u_x", 0.0)   # nonlinear (note: for u*u_x)
        self.d1 = coeffs.get("u_x", 0.0)     # linear advection term
        self.a  = coeffs.get("u_xx", 0.0)    # second derivative
        self.b  = coeffs.get("u_xxxx", 0.0)  # fourth derivative

        dx = L / N
        freqs = np.fft.rfftfreq(N, d=dx)
        k = 2*np.pi*freqs
        self._ik = 1j * k
        Lk = self.a * (-(k**2)) + self.b * (k**4) + self.d1 * (1j * k)
        self._exp = np.exp(self.dt * Lk)
        self._coef = np.where(Lk == 0.0, self.dt, (self._exp - 1.0) / Lk)
        self._alias = (freqs < (2.0/3.0) * freqs.max())  # 2/3 dealias

    def step(self, u):
        u2 = self.c * (u**2)
        uhat = np.fft.rfft(u)
        u2hat = np.fft.rfft(u2) * self._alias
        du2dx_hat = self._ik * u2hat
        unext_hat = self._exp * uhat + self._coef * du2dx_hat
        return np.fft.irfft(unext_hat, n=self.N)

def rollout_rmse(u, L, dt, xi, names, horizon=500):
    T, N = u.shape
    H = min(horizon, T-1)
    coeffs = {name: float(val) for name, val in zip(names, xi)}
    stepper = KS_LearnedStepper(L=L, N=N, dt=dt, coeffs=coeffs)
    pred = np.empty((H+1, N), dtype=float)
    pred[0] = u[0]
    for i in range(H):
        pred[i+1] = stepper.step(pred[i])
    rmse = np.sqrt(np.mean((pred - u[:H+1])**2))
    return float(rmse)


Cell 6 — run SINDy on all datasets, score, and rank

In [14]:
rows = []
files = sorted(glob.glob(os.path.join(RAW_DIR, "L100_N200_dt*_s1_*_seed1_*.h5")))
print("Found datasets:", len(files))

# IMPORTANT: true coefficients for our feature order:
# names = ["u_x", "u_xx", "u_xxxx", "u*u_x"]
# KS form used in your solver: u_t = a u_xx + b u_xxxx + c (u^2)_x,  with a = -1, b = -1, c = -0.5
# Since (u^2)_x = 2 u u_x, the coefficient on (u*u_x) is 2c = -1.0
xi_true = np.array([0.0, -1.0, -1.0, -1.0], dtype=float)
names = ["u_x", "u_xx", "u_xxxx", "u*u_x"]

for path in files:
    with h5py.File(path, "r") as f:
        u = f["u"][:].astype(np.float64)   # (T,N)
        x = f["x"][:].astype(np.float64)
        t = f["t"][:].astype(np.float64)
        attrs = {k: f.attrs[k] for k in f.attrs.keys()}
    dt = float(attrs["dt"])
    Lh = float(attrs["L"])
    tag = os.path.basename(path).replace(".h5","")

    # Build regression data
    y, Theta, _ = build_features(u, x, dt)

    # Fit SINDy (STLSQ)
    xi = stlsq(Theta, y, thresh=1e-3, max_iter=10)

    # Metrics
    supp = support_metrics(xi, xi_true)
    cerr = coeff_errors(xi, xi_true)
    rmse = rollout_rmse(u, L=Lh, dt=dt, xi=xi, names=names, horizon=500)

    rows.append({
        "tag": tag,
        "dt": dt,
        "ic_kind": attrs.get("ic_kind","?"),
        "noise_kind": attrs.get("noise_kind","?"),
        "noise_std": float(attrs.get("noise_std", 0.0)),
        "precision": supp["precision"],
        "recall": supp["recall"],
        "f1": supp["f1"],
        "l1_coeff": cerr["l1"],
        "lmax_coeff": cerr["lmax"],
        "rollout_rmse": rmse,
        "xi_u_x": float(xi[0]),
        "xi_u_xx": float(xi[1]),
        "xi_u_xxxx": float(xi[2]),
        "xi_u_u_x": float(xi[3]),
        "h5": path
    })
    print(f"done: {tag} | F1={supp['f1']:.3f} | L1={cerr['l1']:.3g} | RMSE={rmse:.3g}")

import pandas as pd
df_eval = pd.DataFrame(rows).sort_values(
    by=["f1","l1_coeff","rollout_rmse"], ascending=[False, True, True]
).reset_index(drop=True)

display_cols = ["tag","dt","ic_kind","noise_kind","noise_std","f1","l1_coeff","rollout_rmse",
                "xi_u_x","xi_u_xx","xi_u_xxxx","xi_u_u_x"]
print("\nTOP 12 (best → worst):")
print(df_eval[display_cols].head(12).to_string(index=False))

out_csv = os.path.join(RESULTS_DIR, "phase2_sindy_ranking_corrected.csv")
df_eval.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Found datasets: 72


/tmp/ipykernel_69889/707796585.py:49: RuntimeWarning: invalid value encountered in divide
  self._coef = np.where(Lk == 0.0, self.dt, (self._exp - 1.0) / Lk)


done: L100_N200_dt0.050_s1_gaussian_noise0.000_seed1_gaussian | F1=1.000 | L1=0.0399 | RMSE=0.249
done: L100_N200_dt0.050_s1_gaussian_noise0.000_seed1_laplace | F1=1.000 | L1=0.0399 | RMSE=0.249
done: L100_N200_dt0.050_s1_gaussian_noise0.000_seed1_uniform01 | F1=1.000 | L1=0.0399 | RMSE=0.249
done: L100_N200_dt0.050_s1_gaussian_noise0.020_seed1_gaussian | F1=1.000 | L1=0.0425 | RMSE=0.295
done: L100_N200_dt0.050_s1_gaussian_noise0.020_seed1_laplace | F1=1.000 | L1=0.0383 | RMSE=0.281
done: L100_N200_dt0.050_s1_gaussian_noise0.020_seed1_uniform01 | F1=1.000 | L1=0.0451 | RMSE=0.259
done: L100_N200_dt0.050_s1_gaussian_noise0.050_seed1_gaussian | F1=1.000 | L1=0.0388 | RMSE=0.408
done: L100_N200_dt0.050_s1_gaussian_noise0.050_seed1_laplace | F1=1.000 | L1=0.0379 | RMSE=0.427
done: L100_N200_dt0.050_s1_gaussian_noise0.050_seed1_uniform01 | F1=1.000 | L1=0.0419 | RMSE=0.32
done: L100_N200_dt0.050_s1_gaussian_noise0.100_seed1_gaussian | F1=1.000 | L1=0.04 | RMSE=0.504
done: L100_N200_dt0.050

Cell 7 — (optional) group summaries to see trends

In [15]:
# Average metrics by dt and noise level
agg = (df_eval
       .groupby(["dt","noise_std"])
       [["f1","l1_coeff","rollout_rmse"]]
       .mean()
       .reset_index()
       .sort_values(["dt","noise_std"]))
print(agg.to_string(index=False))


  dt  noise_std  f1  l1_coeff  rollout_rmse
0.05       0.00 1.0  0.035289      0.415973
0.05       0.02 1.0  0.039184      0.517614
0.05       0.05 1.0  0.038353      0.596687
0.05       0.10 1.0  0.040266      0.687193
0.10       0.00 1.0  0.112751      0.681767
0.10       0.02 1.0  0.108823      0.874806
0.10       0.05 1.0  0.108632      0.966878
0.10       0.10 1.0  0.108817      1.032449
